In [1]:
# ============================================================
# 1. CONFIGURAÇÃO E CARGA
# ============================================================
from _setup import *
import unicodedata
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from enem_pipeline.config import (
    PROJECT_ROOT, DATA_DIR, METADATA_DIR, CONSUMO_DIR,
    UF_PROCESSAMENTO, ANO_INICIAL, ANO_FINAL
)
from utils.features_names import read_allfiles,read_allfiles2
from utils.eda import *
import time

warnings.filterwarnings("ignore", category=FutureWarning, module="seaborn")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

# ============================================================
# PARÂMETROS DA ANÁLISE TRANSVERSAL
# ============================================================
UF_PROCESSAMENTO = "PA"



out = read_allfiles2(UF_PROCESSAMENTO,'ESC')
painel = out["painel"]["data"]
faixa = out["faixa"]["data"]
grupo = out["grupo"]["data"]
shape = out["shape"]["data"]
dsp   = out['dsp']["data"]
print("=" * 80)
print(f"UF: {UF_PROCESSAMENTO}")
print(f"Painel: {painel.shape[0]:,} linhas × {painel.shape[1]} colunas")
print(f"Faixa:  {faixa.shape[0]:,} linhas × {faixa.shape[1]} colunas")
print(f"Grupo:  {grupo.shape[0]:,} linhas × {grupo.shape[1]} colunas")
print(f"Dsp:    {dsp.shape[0]:,} linhas × {dsp.shape[1]} colunas")

UF: PA
Painel: 3,625 linhas × 82 colunas
Faixa:  38,335 linhas × 7 colunas
Grupo:  186,449 linhas × 16 colunas
Dsp:    250,436 linhas × 17 colunas


In [2]:
def descrever_dataframe(df,nome_df='df',max_opcoes=15, colunas_excecao=None):
    """
    Lista as variáveis de um DataFrame com tipo, total não nulo
    e características (opções, se for texto).

    Parâmetros
    ----------
    df : pd.DataFrame
        DataFrame a ser descrito.
    max_opcoes : int
        Quantidade máxima de opções a exibir por variável categórica.
    colunas_excecao : list[str], opcional
        Colunas cujos valores NÃO serão listados (só mostra o total).
        Por padrão, qualquer coluna com 'MUNICIPIO' no nome é omitida.
    """
    # Mapa de siglas para expandir automaticamente
    SIGLAS = {
        "CN": "Ciências da Natureza",
        "CH": "Ciências Humanas",
        "LC": "Linguagens e Códigos",
        "MT": "Matemática",
    }

    if colunas_excecao is None:
        colunas_excecao = []

    def _omitir(col):
        return col in colunas_excecao or "MUNICIPIO" in col.upper()

    def _fmt_sigla(v):
        return f"{v} ({SIGLAS[v]})" if v in SIGLAS else v
    print("=" * 100)

    print(f"# Dataframe: {nome_df}")
    print(f"  Shape: {df.shape[0]} linhas × {df.shape[1]} colunas\n")
    cab = f"{'#':<3} {'Variável':<32} {'Tipo':<12} {'Total':>7}  Características"
    print(cab)
    print("-" * 100)

    for i, col in enumerate(df.columns):
        s = df[col]
        tipo = str(s.dtype)
        total = int(s.notna().sum())

        # 1) Colunas de exceção (municípios, códigos etc.)
        if _omitir(col):
            desc = f"(valores omitidos — {s.nunique()} únicos)"

        # 2) Numérico
        elif pd.api.types.is_numeric_dtype(s) and total > 0:
            desc = (f"min={s.min():.2f}, max={s.max():.2f}, "
                    f"média={s.mean():.2f}")

        # 3) Booleano
        elif pd.api.types.is_bool_dtype(s):
            vc = s.value_counts(dropna=False).to_dict()
            desc = ", ".join(f"{k}={v}" for k, v in vc.items())

        # 4) Texto / categórica
        else:
            unicos = sorted(s.dropna().astype(str).unique(),key=len,reverse=True)
            n = len(unicos)
            if n <= max_opcoes:
                desc = ", ".join(_fmt_sigla(u) for u in unicos)
            else:
                amostra = ", ".join(_fmt_sigla(u) for u in unicos[:max_opcoes])
                desc = f"{amostra} ... (+{n - max_opcoes} opções; {n} únicos)"

        print(f"{i:<3} {col:<32} {tipo:<12} {total:>7}  {desc}")

In [3]:
descrever_dataframe(dsp,'dsp')

# Dataframe: dsp
  Shape: 250436 linhas × 17 colunas

#   Variável                         Tipo           Total  Características
----------------------------------------------------------------------------------------------------
0   ano                              Int64         250436  min=2001.00, max=2023.00, média=2013.46
1   CO_MUNICIPIO_TERRITORIO          string        250436  (valores omitidos — 144 únicos)
2   NO_MUNICIPIO_TERRITORIO          str           250436  (valores omitidos — 144 únicos)
3   TIPO_REFERENCIA_TERRITORIAL      str           250436  ESC
4   sexo                             str           250436  MASCULINO, FEMININO
5   faixa_etaria                     str           250436  10, 12, 13, 20, 11, 14, 16, 15, 17, 18, 19, 7, 4, 8, 5 ... (+5 opções; 20 únicos)
6   disciplina                       str           250436  OBJETIVA, REDACAO, CN (Ciências da Natureza), CH (Ciências Humanas), LC (Linguagens e Códigos), MT (Matemática)
7   n_participantes                

In [9]:
dsp[(dsp['NO_MUNICIPIO_TERRITORIO']=='ABAETETUBA')& (dsp['ano']==2001) &(dsp['disciplina']=='REDACAO')]

,ano,CO_MUNICIPIO_TERRITORIO,NO_MUNICIPIO_TERRITORIO,TIPO_REFERENCIA_TERRITORIAL,sexo,faixa_etaria,disciplina,n_participantes,n_validos,n_zeros,media,media_positivas,mediana,std,q1,q3,base_origem
0,2001,1500107,ABAETETUBA,ESC,MASCULINO,7,REDACAO,12,12,6,22.500000,45.000000,12.50,25.915071,0.000,38.125,enem
1,2001,1500107,ABAETETUBA,ESC,MASCULINO,4,REDACAO,29,29,13,26.551724,48.125000,27.50,25.984317,0.000,50.000,enem
5,2001,1500107,ABAETETUBA,ESC,MASCULINO,8,REDACAO,7,7,4,17.142857,40.000000,0.00,21.526562,0.000,37.500,enem
6,2001,1500107,ABAETETUBA,ESC,FEMININO,13,REDACAO,6,6,2,30.000000,45.000000,36.25,25.641763,8.125,41.875,enem
7,2001,1500107,ABAETETUBA,ESC,FEMININO,10,REDACAO,9,9,2,38.611111,49.642857,42.50,26.372071,37.500,45.000,enem
8,2001,1500107,ABAETETUBA,ESC,FEMININO,20,REDACAO,1,1,0,52.500000,52.500000,52.50,NaN,52.500,52.500,enem
9,2001,1500107,ABAETETUBA,ESC,MASCULINO,20,REDACAO,1,1,0,62.500000,62.500000,62.50,NaN,62.500,62.500,enem
15,2001,1500107,ABAETETUBA,ESC,FEMININO,7,REDACAO,21,21,13,17.976190,47.187500,0.00,24.502794,0.000,42.500,enem
16,2001,1500107,ABAETETUBA,ESC,FEMININO,1,REDACAO,10,10,3,40.500000,57.857143,45.00,30.974900,9.375,61.875,enem
18,2001,1500107,ABAETETUBA,ESC,MASCULINO,11,REDACAO,7,7,1,45.714286,53.333333,52.50,24.096779,40.000,56.250,enem


In [13]:
descrever_dataframe(painel,'painel')
descrever_dataframe(grupo,'grupo')
descrever_dataframe(faixa,'faixa')
descrever_dataframe(dsp,'dsp')

# Dataframe: painel
  Shape: 3625 linhas × 82 colunas

#   Variável                         Tipo           Total  Características
----------------------------------------------------------------------------------------------------
0   ano                              Int64           3625  min=1998.00, max=2025.00, média=2012.67
1   CO_MUNICIPIO_TERRITORIO          string          3625  (valores omitidos — 144 únicos)
2   NO_MUNICIPIO_TERRITORIO          str             3625  (valores omitidos — 144 únicos)
3   TIPO_REFERENCIA_TERRITORIAL      str             3625  ESC
4   n_participantes                  int64           3625  min=1.00, max=19261.00, média=343.08
5   n_masculino                      Int64           3337  min=0.00, max=8466.00, média=132.20
6   n_feminino                       Int64           3337  min=0.00, max=10795.00, média=195.74
7   n_faixa_etaria_informada         int64           3625  min=0.00, max=19261.00, média=302.40
8   pct_masculino                    Float